# CreditGuard: Phase 2 - Model Evaluation, Calibration & Threshold Optimization

This notebook unblinds the untouched test set (`X_test`, `y_test`) to compute out-of-sample metrics, evaluate calibration, optimize the decision threshold for financial loss, and select the production model.

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import json
import pandas as pd
from src.config import BEST_MODEL_PATH, MODEL_METRICS_PATH, OPTIMAL_THRESHOLD_PATH
from src.models.persistence import ModelPersistenceManager

persistence = ModelPersistenceManager()
best_pipe = persistence.load_pipeline()
threshold_info = persistence.load_threshold_config()

with open(MODEL_METRICS_PATH, 'r') as f:
    metrics_data = json.load(f)

comparison_df = pd.DataFrame(metrics_data['comparison'])
print('Model Comparison Benchmark Table:')
comparison_df[[
    'Model', 'CV_ROC_AUC', 'Test_ROC_AUC', 'Test_PR_AUC',
    'Test_Precision', 'Test_Recall', 'Test_F1', 'Total_Cost_Default_Thresh',
    'Optimal_Threshold', 'Total_Cost_Optimal_Thresh', 'Financial_Cost_Savings_Pct'
]]

## 1. Cost-Optimal Decision Threshold Strategy
Defaulting borrower loss ($5,000) vs rejected good applicant loss ($1,000).

In [ ]:
print(f'Champion Model: {metrics_data["best_model"]}')
print(f'Default Decision Threshold: 0.5000')
print(f'Optimal Decision Threshold: {threshold_info["selected_threshold"]:.4f}')
print(f'Estimated Portfolio Risk Cost Reduction: {threshold_info["metadata"]["cost_savings_pct"]:.2f}%')

## 2. Feature Importance Analysis

In [ ]:
from src.models.trainer import ModelTrainer
trainer = ModelTrainer()
importances = trainer._extract_feature_importances(best_pipe)
if importances:
    imp_df = pd.DataFrame(list(importances.items()), columns=['Feature', 'Importance']).head(15)
    print('Top 15 Most Influential Features in XGBoost Champion Pipeline:')
    print(imp_df.to_string(index=False))
else:
    print('Feature importances not available.')